# Exploratory Data Analysis — CT Brain Image Classification

**Dataset:** 256 CT Brain Scans (Aneurysm, Cancer, Tumor)

**Notebook ini menghasilkan visualisasi untuk paper IEEE:**
- **Fig. 1.** Proposed Experimental Workflow
- **Fig. 2.** Dataset Samples (a) and Class Distribution (b)
- **TABLE I.** Dataset Partitioning Matrices Across Scenarios

In [ ]:
import os
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
from matplotlib.patches import ConnectionPatch
from PIL import Image
from collections import Counter

warnings.filterwarnings('ignore')

matplotlib.rcParams.update({
    'font.size': 10,
    'font.family': 'DejaVu Sans',
    'axes.titlesize': 12,
    'axes.labelsize': 11,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.05
})

print(f"Python: {sys.version.split()[0]}")
print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")

---
## 1. Data Loading

In [ ]:
CSV_PATH = '../dataset/ct_brain.csv'
IMG_BASE = '../dataset/files'

DATASET_AVAILABLE = os.path.isfile(CSV_PATH)

if DATASET_AVAILABLE:
    df = pd.read_csv(CSV_PATH)
    df['full_path'] = IMG_BASE + df['jpg']
    label_mapping = {label: idx for idx, label in enumerate(df['type'].unique())}
    idx_to_label = {v: k for k, v in label_mapping.items()}
    df['label'] = df['type'].map(label_mapping)
    print(f'Dataset loaded: {len(df)} images')
    print(f'Label mapping: {label_mapping}')
    print(f'\nClass distribution:')
    for cls, count in df['type'].value_counts().items():
        print(f'  {cls:12s}: {count:3d}  ({count/len(df)*100:5.1f}%)')
    print(f'  {"TOTAL":12s}: {len(df):3d}')
    display(df.head())
else:
    print(f'WARNING: Dataset CSV not found at {os.path.abspath(CSV_PATH)}')
    print('Running with hardcoded class distribution from training notebook outputs.')
    print('Place the dataset to render sample CT images (Figure 2a).')
    CLASS_COUNTS = {'aneurysm': 89, 'cancer': 89, 'tumor': 78}
    label_mapping = {'aneurysm': 0, 'tumor': 1, 'cancer': 2}
    idx_to_label = {0: 'aneurysm', 1: 'tumor', 2: 'cancer'}

---
## 2. Figure 1 — Proposed Experimental Workflow

Block diagram of the full research pipeline: Data Acquisition → Preprocessing → Splitting → Model Architectures → Hyperparameter Tuning → Evaluation.

In [ ]:
def draw_block(ax, xy, width, height, text, color, text_color='black', fontsize=9, linewidth=2):
    """Draw a rounded FancyBbox block with text centered inside."""
    rect = FancyBboxPatch(
        xy, width, height,
        boxstyle="round,pad=0.15,rounding_size=6",
        facecolor=color, edgecolor='#2c3e50', linewidth=linewidth, alpha=0.92
    )
    ax.add_patch(rect)
    rx, ry = xy[0] + width / 2, xy[1] + height / 2
    lines = text.split('\n')
    for i, line in enumerate(lines):
        y_offset = (len(lines) - 1) * 0.012 - i * 0.024
        if len(lines) == 1:
            y_offset = 0
        ax.text(rx, ry + y_offset, line, ha='center', va='center', fontsize=fontsize,
                fontweight='bold', color=text_color)

def draw_arrow(ax, start_xy, end_xy, color='#34495e', lw=2.2, style='->'):
    """Draw a straight arrow from start_xy to end_xy."""
    ax.annotate('', xy=end_xy, xytext=start_xy,
                arrowprops=dict(arrowstyle=style, color=color, lw=lw,
                                connectionstyle='arc3,rad=0'))

def draw_split_arrow(ax, start_xy, end_left, end_right, color='#34495e', lw=2.0):
    """Draw a branching arrow (one-to-two)."""
    mid_y = (start_xy[1] + end_left[1]) / 2
    draw_arrow(ax, start_xy, (start_xy[0], mid_y), color, lw)
    draw_arrow(ax, (start_xy[0], mid_y), (end_left[0], mid_y), color, lw)
    draw_arrow(ax, (end_left[0], mid_y), end_left, color, lw)
    draw_arrow(ax, (start_xy[0], mid_y), (end_right[0], mid_y), color, lw)
    draw_arrow(ax, (end_right[0], mid_y), end_right, color, lw)
    ax.text(start_xy[0] + 0.5, mid_y + 0.025, 'Train/Val/Test', ha='center', va='bottom',
            fontsize=7.5, fontstyle='italic', color='#2c3e50')

def draw_merge_arrows(ax, tops, bottom_center, color='#34495e', lw=2.0):
    """Draw merging arrows (three-to-one)."""
    mid_y = (tops[0][1] + bottom_center[1]) / 2
    for top in tops:
        draw_arrow(ax, top, (top[0], mid_y), color, lw)
        draw_arrow(ax, (top[0], mid_y), (bottom_center[0], mid_y), color, lw)
    draw_arrow(ax, (bottom_center[0], mid_y), bottom_center, color, lw)

# ─── Build Figure ───
fig, ax = plt.subplots(1, 1, figsize=(14, 8))
ax.set_xlim(0, 14)
ax.set_ylim(0, 8)
ax.set_aspect('equal')
ax.axis('off')

# Color palette (professional, accessible)
C_BLUE   = '#3498db'
C_GREEN  = '#2ecc71'
C_ORANGE = '#e67e22'
C_RED    = '#e74c3c'
C_PURPLE = '#9b59b6'
C_TEAL   = '#1abc9c'
C_GRAY   = '#95a5a6'
C_DARK   = '#2c3e50'
C_YELLOW = '#f1c40f'

# ─── Row 1: Input → Preprocessing ───
draw_block(ax, (0.5, 6.5), 2.8, 0.85, 'CT Scan Input\n(256 Brain CT Images)', C_BLUE, 'white', 9)
draw_arrow(ax, (3.3, 6.925), (4.5, 6.925))
draw_block(ax, (4.5, 6.5), 2.8, 0.85, 'Preprocessing\nResize + Normalization', C_GREEN, 'white', 9)

# ─── Row 2: Data Splitting ───
draw_arrow(ax, (5.9, 6.5), (5.9, 5.9))
draw_block(ax, (4.0, 5.0), 3.8, 0.85,
           'Data Splitting\nScenario 1: 80/10/10  |  Scenario 2: 70/20/10',
           C_ORANGE, 'white', 8.5)

# ─── Row 3: Three Model Architectures ───
model_y = 3.2
model_w, model_h = 3.2, 1.0
gap = 0.8
x_positions = [0.5, 5.0, 9.5]
model_colors = [C_TEAL, C_PURPLE, C_RED]
model_names = ['VGG16\n(CNN Baseline)', 'ResNet50\n(Residual CNN)', 'Vision Transformer\n(ViT-B/16)']
model_tops = []

for x, col, name in zip(x_positions, model_colors, model_names):
    draw_block(ax, (x, model_y), model_w, model_h, name, col, 'white', 8.5)
    model_tops.append((x + model_w/2, model_y + model_h))

# Arrows from splitting to models (branching)
split_center = (5.9, 5.0)
draw_merge_arrows(ax, [split_center], split_center, C_DARK, 0.01)  # noop
mid_connection_y = 4.35
draw_arrow(ax, split_center, (split_center[0], mid_connection_y), C_DARK, 2.0)
draw_arrow(ax, (split_center[0], mid_connection_y), (x_positions[0] + model_w/2, mid_connection_y), C_DARK, 2.0)
draw_arrow(ax, (x_positions[0] + model_w/2, mid_connection_y), model_tops[0], C_DARK, 2.0)
draw_arrow(ax, (split_center[0], mid_connection_y), (x_positions[1] + model_w/2, mid_connection_y), C_DARK, 2.0)
draw_arrow(ax, (x_positions[1] + model_w/2, mid_connection_y), model_tops[1], C_DARK, 2.0)
draw_arrow(ax, (split_center[0], mid_connection_y), (x_positions[2] + model_w/2, mid_connection_y), C_DARK, 2.0)
draw_arrow(ax, (x_positions[2] + model_w/2, mid_connection_y), model_tops[2], C_DARK, 2.0)

# ─── Row 4: Hyperparameter Tuning ───
hp_y = 1.5
merge_connection_y = 2.3
model_bots = [(x + model_w/2, model_y) for x in x_positions]
for bot in model_bots:
    draw_arrow(ax, bot, (bot[0], merge_connection_y), C_DARK, 2.0)
    draw_arrow(ax, (bot[0], merge_connection_y), (6.9, merge_connection_y), C_DARK, 2.0)
draw_arrow(ax, (6.9, merge_connection_y), (6.9, hp_y + 0.85), C_DARK, 2.0)

draw_block(ax, (4.5, hp_y), 4.8, 0.85,
           'Hyperparameter Tuning\nLearning Rate: 1e-2 (Aggressive)  |  1e-4 (Conservative)',
           C_YELLOW, C_DARK, 8.5)

# ─── Row 5: Output ───
draw_arrow(ax, (6.9, hp_y), (6.9, 0.5))
draw_block(ax, (4.0, -0.15), 5.8, 0.85,
           'Output: Classification\nAneurysm  |  Cancer  |  Tumor',
           C_DARK, 'white', 9.5)

# ─── Legend / Phase Labels (left side) ───
phase_labels = [
    (0.0, 6.925, 'Phase 1'),
    (0.0, 5.425, 'Phase 2'),
    (0.0, 3.7, 'Phase 3'),
    (0.0, 1.925, 'Phase 4'),
    (0.0, 0.275, 'Phase 5'),
]
for x, y, label in phase_labels:
    ax.text(x, y, label, fontsize=7.5, fontweight='bold', color='#7f8c8d',
            va='center', ha='right', fontstyle='italic',
            bbox=dict(boxstyle='round,pad=0.2', facecolor='#ecf0f1', edgecolor='none', alpha=0.7))

fig.suptitle('Fig. 1. Proposed experimental workflow encompassing data preprocessing,\n'
             'model selection, hyperparameter tuning, and evaluation.',
             fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('fig1_workflow.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print('Saved: fig1_workflow.png')

---
## 3. Figure 2(a) — Sample CT Scan Images

Menampilkan contoh citra dari masing-masing kelas (Aneurysm, Cancer, Tumor) untuk menunjukkan kompleksitas visual dan kemiripan antar kelas.

In [ ]:
SAMPLE_PATH = '../dataset/files'
OUTPUT_DIR = '.'

if DATASET_AVAILABLE and os.path.isdir(SAMPLE_PATH):
    classes = ['aneurysm', 'cancer', 'tumor']
    n_samples = 4

    fig, axes = plt.subplots(3, n_samples, figsize=(2.8 * n_samples, 7))

    for row, cls in enumerate(classes):
        cls_dir = os.path.join(SAMPLE_PATH, cls)
        if not os.path.isdir(cls_dir):
            continue
        images = sorted(os.listdir(cls_dir))[:n_samples]
        for col, img_name in enumerate(images):
            img_path = os.path.join(cls_dir, img_name)
            img = Image.open(img_path).convert('RGB')
            axes[row, col].imshow(img, cmap='gray')
            axes[row, col].axis('off')
            if col == 0:
                axes[row, col].set_ylabel(cls.capitalize(), fontsize=11, fontweight='bold',
                                         rotation=0, labelpad=45, va='center')
            if row == 0:
                axes[row, col].set_title(f'Sample {col+1}', fontsize=9.5, fontweight='bold')

    fig.suptitle('Fig. 2(a). Sample brain CT scans representing Aneurysm, Cancer, and Tumor classes.',
                 fontsize=12, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'fig2a_samples.png'), dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()
    print('Saved: fig2a_samples.png')
else:
    print('WARNING: Dataset not available. Cannot render sample CT images.')
    print('Place the dataset at ../dataset/ and re-run this cell.')
    print('\nExpected structure:')
    print('  ../dataset/files/aneurysm/*.jpg')
    print('  ../dataset/files/cancer/*.jpg')
    print('  ../dataset/files/tumor/*.jpg')

---
## 4. Figure 2(b) — Class Distribution

Bar chart distribusi jumlah data per kelas untuk menunjukkan keseimbangan dataset.

In [ ]:
if DATASET_AVAILABLE:
    class_counts = df['type'].value_counts().to_dict()
    class_counts = {k: class_counts.get(k, 0) for k in ['aneurysm', 'cancer', 'tumor']}
else:
    class_counts = {'aneurysm': 89, 'cancer': 89, 'tumor': 78}

total = sum(class_counts.values())

fig, ax = plt.subplots(figsize=(7, 4.5))

classes = list(class_counts.keys())
counts = list(class_counts.values())
colors_bars = ['#3498db', '#e74c3c', '#2ecc71']

bars = ax.bar(classes, counts, color=colors_bars, edgecolor='#2c3e50', linewidth=1.2, width=0.55, alpha=0.88)

for bar, count in zip(bars, counts):
    pct = count / total * 100
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1.2,
            f'{count}\n({pct:.1f}%)', ha='center', va='bottom', fontsize=10, fontweight='bold', color='#2c3e50')

ax.set_ylim(0, max(counts) * 1.18)
ax.set_ylabel('Number of Images', fontsize=11, fontweight='bold')
ax.set_xticks(range(len(classes)))
ax.set_xticklabels([c.capitalize() for c in classes], fontsize=11, fontweight='bold')
ax.set_title(f'Class Distribution (N = {total})', fontsize=12, fontweight='bold', pad=12)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='y', alpha=0.3, linestyle='--')

fig.suptitle('Fig. 2(b). Class distribution across the dataset.',
             fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'fig2b_distribution.png'), dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print('Saved: fig2b_distribution.png')

---
## 5. Figure 2 (Combined) — Full Figure for Paper

Menggabungkan (a) Sample Images dan (b) Class Distribution dalam satu figure IEEE-style.

In [ ]:
if DATASET_AVAILABLE and os.path.isdir(SAMPLE_PATH):
    fig = plt.figure(figsize=(14, 5.5))

    # ─── Sub-figure (a): Sample Images ───
    gs = fig.add_gridspec(3, 5, left=0.05, right=0.52, top=0.88, bottom=0.08, hspace=0.15, wspace=0.05)

    classes = ['aneurysm', 'cancer', 'tumor']
    n_samples = 4

    for row, cls in enumerate(classes):
        cls_dir = os.path.join(SAMPLE_PATH, cls)
        images = sorted(os.listdir(cls_dir))[:n_samples]
        for col, img_name in enumerate(images):
            ax_img = fig.add_subplot(gs[row, col])
            img_path = os.path.join(cls_dir, img_name)
            img = Image.open(img_path).convert('RGB')
            ax_img.imshow(img, cmap='gray')
            ax_img.axis('off')
            if col == 0:
                ax_img.set_ylabel(cls.capitalize(), fontsize=11, fontweight='bold',
                                  rotation=0, labelpad=40, va='center')
        # Add annotation for 4th column
        ax_label = fig.add_subplot(gs[row, n_samples])
        ax_label.axis('off')

    fig.text(0.285, 0.92, '(a) Sample CT Scans', ha='center', fontsize=11, fontweight='bold')

    # ─── Sub-figure (b): Class Distribution ───
    ax_bar = fig.add_axes([0.58, 0.18, 0.38, 0.65])

    bars2 = ax_bar.bar(classes, counts, color=colors_bars, edgecolor='#2c3e50', linewidth=1.2, width=0.5, alpha=0.88)
    for bar, count in zip(bars2, counts):
        pct = count / total * 100
        ax_bar.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1.2,
                    f'{count} ({pct:.1f}%)', ha='center', va='bottom', fontsize=10.5, fontweight='bold', color='#2c3e50')

    ax_bar.set_ylim(0, max(counts) * 1.16)
    ax_bar.set_ylabel('Number of Images', fontsize=10, fontweight='bold')
    ax_bar.set_xticks(range(len(classes)))
    ax_bar.set_xticklabels([c.capitalize() for c in classes], fontsize=11, fontweight='bold')
    ax_bar.spines['top'].set_visible(False)
    ax_bar.spines['right'].set_visible(False)
    ax_bar.grid(axis='y', alpha=0.3, linestyle='--')

    fig.text(0.77, 0.92, '(b) Class Distribution', ha='center', fontsize=11, fontweight='bold')

    fig.suptitle('Fig. 2. (a) Sample brain CT scans representing Aneurysm, Cancer, and Tumor classes. ',
                 '\n(b) Class distribution across the dataset.',
                 fontsize=12, fontweight='bold', x=0.5, y=0.97, va='top')

    plt.savefig(os.path.join(OUTPUT_DIR, 'fig2_combined.png'), dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()
    print('Saved: fig2_combined.png')
else:
    print('WARNING: Combined Figure 2 requires dataset. Skipping.')
    print('Figure 2(b) bar chart was already generated above.')

---
## 6. TABLE I — Dataset Partitioning Matrix

Tabel ringkas IEEE-style yang menunjukkan jumlah citra untuk Train, Validation, dan Test pada Skenario 1 (80:10:10) dan Skenario 2 (70:20:10). Test set identik di kedua skenario.

In [ ]:
from sklearn.model_selection import train_test_split

if DATASET_AVAILABLE:
    def get_split_counts(df, scenario):
        if scenario == 1:
            train_ratio, val_ratio = 0.8, 0.1
        elif scenario == 2:
            train_ratio, val_ratio = 0.7, 0.2
        else:
            raise ValueError('Invalid scenario')
        trainval_df, test_df = train_test_split(df, test_size=0.1, random_state=42, stratify=df['label'])
        val_size = val_ratio / (train_ratio + val_ratio)
        train_df, val_df = train_test_split(trainval_df, test_size=val_size, random_state=42, stratify=trainval_df['label'])
        return len(train_df), len(val_df), len(test_df)

    train1, val1, test1 = get_split_counts(df, 1)
    train2, val2, test2 = get_split_counts(df, 2)
else:
    train1, val1, test1 = 204, 26, 26
    train2, val2, test2 = 178, 52, 26

def get_per_class_split(df_full, scenario):
    """Return per-class breakdown of train/val/test sets (requires dataset)."""
    if not DATASET_AVAILABLE:
        return None
    if scenario == 1:
        train_ratio, val_ratio = 0.8, 0.1
    elif scenario == 2:
        train_ratio, val_ratio = 0.7, 0.2
    else:
        raise ValueError('Invalid scenario')
    trainval_df, test_df = train_test_split(df_full, test_size=0.1, random_state=42, stratify=df_full['label'])
    val_size = val_ratio / (train_ratio + val_ratio)
    train_df, val_df = train_test_split(trainval_df, test_size=val_size, random_state=42, stratify=trainval_df['label'])
    breakdown = {}
    for name, subset in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
        breakdown[name] = subset['type'].value_counts().to_dict()
    return breakdown

# ─── Build IEEE-style table ───
fig, ax = plt.subplots(figsize=(8.5, 3.2))
ax.axis('off')

table_data = [
    ['1', '80 : 10 : 10', str(train1), str(val1), str(test1)],
    ['2', '70 : 20 : 10', str(train2), str(val2), str(test2)],
]
col_labels = ['Scenario', 'Split Ratio', 'Train Size', 'Val Size', 'Test Size']

table = ax.table(
    cellText=table_data,
    colLabels=col_labels,
    cellLoc='center',
    loc='center',
    colWidths=[0.12, 0.25, 0.18, 0.18, 0.18],
)

table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.0, 1.7)

for (row, col), cell in table.get_celld().items():
    cell.set_edgecolor('#2c3e50')
    cell.set_linewidth(1.2)
    if row == 0:
        cell.set_facecolor('#2c3e50')
        cell.set_text_props(color='white', fontweight='bold', fontsize=9.5)
        cell.set_edgecolor('#2c3e50')
    else:
        cell.set_facecolor('#f8f9fa' if row % 2 == 1 else 'white')
        cell.set_text_props(color='#2c3e50', fontweight='normal')

ax.text(0.5, 0.93, 'TABLE I. DATASET PARTITIONING MATRICES ACROSS SCENARIOS',
        transform=ax.transAxes, ha='center', va='top', fontsize=11, fontweight='bold')

ax.text(0.5, -0.02, 'Note: The test set contains exactly 26 identical images across both scenarios (9 aneurysm, 8 tumor, 9 cancer).',
        transform=ax.transAxes, ha='center', va='top', fontsize=8, fontstyle='italic', color='#555555')

plt.savefig(os.path.join(OUTPUT_DIR, 'table1_partitioning.png'), dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print('Saved: table1_partitioning.png')
print(f'\nSummary:')
print(f'  Scenario 1 (80:10:10) → Train: {train1}, Val: {val1}, Test: {test1}  (Total: {train1+val1+test1})')
print(f'  Scenario 2 (70:20:10) → Train: {train2}, Val: {val2}, Test: {test2}  (Total: {train2+val2+test2})')

---
## 7. TABLE I (Extended) — Per-Class Breakdown

Distribusi per kelas untuk Train, Val, dan Test di masing-masing skenario (dibutuhkan untuk justifikasi stratified split).

In [ ]:
if DATASET_AVAILABLE:
    bd1 = get_per_class_split(df, 1)
    bd2 = get_per_class_split(df, 2)

    # Print Scenario 1 breakdown
    print('\n' + '='*65)
    print('  PER-CLASS BREAKDOWN — Scenario 1 (80:10:10)')
    print('='*65)
    row_fmt = '{:<12s} {:>8s} {:>8s} {:>8s} {:>8s} {:>10s}'
    print(row_fmt.format('Class', 'Train', 'Val', 'Test', 'Total', 'Total %'))
    print('-'*65)
    for cls in ['aneurysm', 'cancer', 'tumor']:
        tr, va, te = bd1['Train'].get(cls, 0), bd1['Val'].get(cls, 0), bd1['Test'].get(cls, 0)
        subtotal = tr + va + te
        pct = subtotal / total * 100
        print(row_fmt.format(cls.capitalize(), str(tr), str(va), str(te), str(subtotal), f'{pct:.1f}%'))
    print('-'*65)
    tr = sum(bd1['Train'].values())
    va = sum(bd1['Val'].values())
    te = sum(bd1['Test'].values())
    print(row_fmt.format('TOTAL', str(tr), str(va), str(te), str(tr+va+te), '100.0%'))

    # Print Scenario 2 breakdown
    print('\n' + '='*65)
    print('  PER-CLASS BREAKDOWN — Scenario 2 (70:20:10)')
    print('='*65)
    print(row_fmt.format('Class', 'Train', 'Val', 'Test', 'Total', 'Total %'))
    print('-'*65)
    for cls in ['aneurysm', 'cancer', 'tumor']:
        tr, va, te = bd2['Train'].get(cls, 0), bd2['Val'].get(cls, 0), bd2['Test'].get(cls, 0)
        subtotal = tr + va + te
        pct = subtotal / total * 100
        print(row_fmt.format(cls.capitalize(), str(tr), str(va), str(te), str(subtotal), f'{pct:.1f}%'))
    print('-'*65)
    tr = sum(bd2['Train'].values())
    va = sum(bd2['Val'].values())
    te = sum(bd2['Test'].values())
    print(row_fmt.format('TOTAL', str(tr), str(va), str(te), str(tr+va+te), '100.0%'))
else:
    print('WARNING: Dataset not available. Cannot compute per-class breakdown.')
    print('\nEstimated per-class test set counts (from training notebook outputs):')
    print('  Aneurysm: 9,  Tumor: 8,  Cancer: 9  (Total: 26)')

---
## 8. Bonus EDA — Pixel Intensity Analysis

Analisis distribusi intensitas piksel per kelas untuk karakterisasi statistik citra medis.

In [ ]:
if DATASET_AVAILABLE and os.path.isdir(SAMPLE_PATH):
    classes = ['aneurysm', 'cancer', 'tumor']
    colors_hist = {'aneurysm': '#3498db', 'cancer': '#e74c3c', 'tumor': '#2ecc71'}

    pixel_stats = {}

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))

    for i, cls in enumerate(classes):
        cls_dir = os.path.join(SAMPLE_PATH, cls)
        images = sorted(os.listdir(cls_dir))[:30]
        all_pixels = []
        for img_name in images:
            img_path = os.path.join(cls_dir, img_name)
            img = Image.open(img_path).convert('L')
            arr = np.array(img).flatten()
            all_pixels.extend(arr)
        all_pixels = np.array(all_pixels)
        pixel_stats[cls] = {'mean': np.mean(all_pixels), 'std': np.std(all_pixels),
                           'min': np.min(all_pixels), 'max': np.max(all_pixels)}

        axes[i].hist(all_pixels, bins=80, color=colors_hist[cls], edgecolor='#2c3e50',
                    alpha=0.8, linewidth=0.5, density=True)
        axes[i].axvline(np.mean(all_pixels), color='#2c3e50', linestyle='--', linewidth=1.5,
                       label=f'Mean: {np.mean(all_pixels):.1f}')
        axes[i].set_title(f'{cls.capitalize()}\nμ={np.mean(all_pixels):.1f}, σ={np.std(all_pixels):.1f}',
                          fontsize=11, fontweight='bold')
        axes[i].set_xlabel('Pixel Intensity', fontsize=9)
        axes[i].set_ylabel('Density', fontsize=9)
        axes[i].legend(fontsize=8, loc='upper right')
        axes[i].spines['top'].set_visible(False)
        axes[i].spines['right'].set_visible(False)

    fig.suptitle('Pixel Intensity Distribution by Class', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'fig3_pixel_intensity.png'), dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()

    print('\nPixel Intensity Statistics:')
    for cls, stats in pixel_stats.items():
        print(f'  {cls.capitalize():12s}  Mean: {stats["mean"]:6.1f}  Std: {stats["std"]:6.1f}  '
              f'Min: {stats["min"]:5.0f}  Max: {stats["max"]:5.0f}')
    print('Saved: fig3_pixel_intensity.png')
else:
    print('WARNING: Dataset not available. Skipping pixel intensity analysis.')

---
## 9. Bonus EDA — Image Dimension / Resolution Analysis

Memeriksa konsistensi dimensi citra di seluruh dataset.

In [ ]:
if DATASET_AVAILABLE and os.path.isdir(SAMPLE_PATH):
    classes = ['aneurysm', 'cancer', 'tumor']
    all_dims = []
    for cls in classes:
        cls_dir = os.path.join(SAMPLE_PATH, cls)
        images = os.listdir(cls_dir)
        for img_name in images:
            img_path = os.path.join(cls_dir, img_name)
            try:
                with Image.open(img_path) as img:
                    all_dims.append((cls, img.size[0], img.size[1]))
            except Exception:
                pass

    dims_df = pd.DataFrame(all_dims, columns=['class', 'width', 'height'])
    display(dims_df.groupby('class').agg(['mean', 'std', 'min', 'max', 'count']).round(1))
    print(f'\nUnique dimension combinations: {dims_df[["width", "height"]].drop_duplicates().shape[0]}')
else:
    print('WARNING: Dataset not available. Skipping dimension analysis.')

---
## 10. Summary — Generated Figures

Ringkasan semua output yang dihasilkan oleh notebook ini.

In [ ]:
import glob

generated = sorted(glob.glob('fig*.png') + glob.glob('table*.png'))
print('Generated files:')
if generated:
    for f in generated:
        size_kb = os.path.getsize(f) / 1024
        print(f'  {f:40s}  {size_kb:7.1f} KB')
else:
    print('  No files generated yet. Run the cells above.')

print(f'\nReady for inclusion in IEEE paper (300 DPI PNG format).')